In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()
from tqdm import tqdm
import pandas as pd
import numpy as np
from pathlib import Path
 
import time
from langchain_nvidia_ai_endpoints import NVIDIAEmbeddings
import pickle


In [ ]:
# client=OpenAI(base_url="https://openrouter.ai/api/v1",api_key=os.getenv("OPENROUTER_API_KEY")) -> Rate limit for free acc next nvidia 
client=NVIDIAEmbeddings(model="nvidia/nemotron-3-embed-1b",api_key=os.getenv("NVIDIA_API_KEY"),trucate="NONE")

In [3]:
# Hit with rate limit 
# result=client.embeddings.create(input=texts_to_embed,model="nvidia/nemotron-3-embed-1b:free",encoding_format="float")

In [3]:
texts_to_embed=["The business of physical artificial intelligence already generates some $10 billion in annual revenue at Nvidia. While that is still a fraction of the company’s overall revenue, which totaled $303 billion in the four quarters ended in July, Chief Executive Jensen Huang says it will grow 10-fold within the next decade."]
res=client.embed_documents(texts=texts_to_embed)

In [4]:
np.asarray(res).shape

(1, 2048)

In [6]:
# client=InferenceClient(model="google/embeddinggemma-300m",token=hf_token)
# client=OpenAI(base_url="https://integrate.api.nvidia.com/v1",api_key=os.getenv("NVIDIA_API_KEY"))
# client=NVIDIAEmbeddings(
#     model="nvidia/nemotron-3-embed-1b",
#     trucate="NONE",
#     api_key=os.getenv("NVIDIA_API_KEY"))

In [2]:
ROOT_DIR=Path.cwd()
DATA_DIR=ROOT_DIR/'data'/'fiqa'
INDEX_DIR=ROOT_DIR/'indexes'/'dense'
PICKLE_PATH=ROOT_DIR/'indexes'/'embeddings.pickle'


In [14]:
def embed_batch(texts:list[str])->np.ndarray:
    """ Embed a batch of texts and return a len(texts),1536 array"""
    response=client.embed_documents(texts=texts)
    return np.asarray(response)

In [13]:
def build_index(doc_texts:list[str],batch:int=256)->np.ndarray:
    """ Create pickle file and save the progess so interrupted sessions can resume"""
    """ Embed the full corpus in batches with progress bar"""
    chunks=[] 
    start_index=0
    wait=1.5
    
    checkpoint_at=10
    if os.path.exists(PICKLE_PATH):
        with open(PICKLE_PATH,"rb") as f:
            data=pickle.load(f)
            chunks=data['embedding']
            start_index=data['next_index']

    batch_counter=0
    for i in tqdm(range(start_index,len(doc_texts),batch),desc="Embedding"):
        chunks.append(embed_batch(doc_texts[i:i+batch]))
        batch_counter+=1
        if(batch_counter%checkpoint_at)==0:
            with open(PICKLE_PATH,"wb") as f:
                pickle.dump({"embedding":chunks,'next_index':i+batch},f)
        time.sleep(wait)
    return np.vstack(chunks)


In [6]:
corpus=pd.read_parquet(DATA_DIR/"corpus.parquet")
doc_ids=corpus['_id'].tolist()
doc_texts=[t.strip() or "[empty document]" for t in corpus['text'].tolist()]

In [3]:
embedding_path=INDEX_DIR/"embeddings.npy"
if embedding_path.exists():
    print("Loading embedding from ",embedding_path)
    doc_embeddings=np.load(embedding_path)

else:
    print(f"Embedding  {len(doc_texts)}")
    doc_embeddings=build_index(doc_texts)
    embedding_path.parent.mkdir(parents=True,exist_ok=True)
    np.save(embedding_path,doc_embeddings)
    print("Embedding saved")

Loading embedding from  d:\Bluebird\Code 1\Agentic RAG\02-hybrid-retrieval\indexes\dense\embeddings.npy


In [4]:
doc_embeddings_normalized=doc_embeddings/np.linalg.norm(doc_embeddings,axis=1,keepdims=True)

In [9]:
doc_embeddings_normalized[0]

array([ 0.22713594,  0.02088267, -0.04532866, ...,  0.07615008,
       -0.02222869,  0.0163064 ], shape=(2048,))

In [10]:
def search_dense(query:str,k:int)->list[tuple[str,float]]:
    query_vec=embed_batch([query])[0]
    query_vec/=np.linalg.norm(query_vec)
    scores=doc_embeddings_normalized @ query_vec
    top_k=np.argsort(-scores)[:k]
    return [(doc_ids[i],float(scores[i])) for i in top_k]

In [16]:
if __name__=="__main__":
    query="where should I park my rainy funds"
    print(f"\nQuery: {query}\n")
    for i,(doc_id,score) in enumerate(search_dense(query,k=5)):
        text=corpus.loc[corpus['_id']==doc_id,'text'].iloc[0]
        print(f"{i} [{score:.3f}] {doc_id} {text}\n")


Query: where should I park my rainy funds

0 [0.643] 551764 Here are some options: Park your money in a bank that works best for you.

1 [0.624] 63569 What to do with your equity?  Leave it alone...

2 [0.604] 450228 Never invest money you need in the short term. As already suggested, park your money in CDs.

3 [0.588] 490397 As per OP's request Slush fund, maybe?

4 [0.576] 215296 If you have the money put it in, and invest the money over a few months. Always keep a modest reserve of cash to give you the flexibility to take advantage of market opportunities.

